# Fiche 6 — Mesurer une segmentation : IoU et Dice

| | |
|---|---|
| 🎯 **Objectif** | Calculer à la main puis avec NumPy l'IoU et le Dice d'une classe, et expliquer pourquoi l'exactitude par pixel trompe |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 5 |
| 🔗 **Dans le projet** | Étape 5 (choix d'une métrique pertinente : IoU, Dice, F1) |

> **L'idée clé.** Pour juger un vêtement, on compare **deux silhouettes** : celle dessinée par l'annotateur et celle prédite par le modèle. L'IoU, c'est la part de leur surface totale qu'elles **partagent**.

## 1. À la main, sur une grille de 4 × 4

Deux silhouettes d'un même vêtement : `V` (vérité, l'annotation) et `P` (prédiction du modèle).

In [1]:
import numpy as np

V = np.array([[0, 1, 1, 0],
              [0, 1, 1, 0],
              [0, 1, 1, 0],
              [0, 0, 0, 0]], dtype=bool)
P = np.array([[0, 0, 0, 0],
              [0, 1, 1, 1],
              [0, 1, 1, 1],
              [0, 0, 0, 0]], dtype=bool)

intersection = (V & P).sum()   # pixels dans les deux
union = (V | P).sum()          # pixels dans l'un OU l'autre
print(f"vérité {V.sum()} px, prédiction {P.sum()} px, communs {intersection}, union {union}")
print(f"IoU  = {intersection} / {union} = {intersection / union:.2f}")
print(f"Dice = 2 × {intersection} / ({V.sum()} + {P.sum()}) = {2 * intersection / (V.sum() + P.sum()):.2f}")

vérité 6 px, prédiction 6 px, communs 4, union 8
IoU  = 4 / 8 = 0.50
Dice = 2 × 4 / (6 + 6) = 0.67


- **IoU** (*Intersection over Union*) : 1 = superposition parfaite, 0 = aucun pixel en commun.
- **Dice** : 2 × communs ÷ (vrais + prédits). C'est exactement le **F1-score** calculé sur les pixels. Il se déduit de
  l'IoU (Dice = 2 IoU / (1 + IoU)) : il classe les vêtements dans le même ordre, avec des valeurs plus hautes.
  On en publie un, et l'autre en complément.

## 2. Sur une vraie image du projet

Avec 18 classes, on range tout dans une **matrice de confusion** : la case (i, j) compte les pixels de classe i
(vérité) prédits en classe j. L'intersection d'une classe est sur la diagonale ; son union = sa ligne + sa colonne
− la diagonale.

In [2]:
import pandas as pd

from segvet import cache, io
from segvet.labels import CLASSES, FRENCH
from segvet.metrics import confusion_matrix, dice_per_class, iou_per_class, pixel_accuracy

pairs = io.load_pairs()
image, mask = pairs[0]
verite, prediction = io.read_mask(mask), cache.read_prediction(image, "jpeg512")
cm = confusion_matrix(verite, prediction)
scores = pd.DataFrame({"classe": [FRENCH[c] for c in CLASSES], "IoU": iou_per_class(cm), "Dice": dice_per_class(cm)})
scores.dropna().round(2)

,classe,IoU,Dice
0,Fond,0.99,0.99
2,Cheveux,0.74,0.85
4,Haut,0.92,0.96
6,Pantalon,0.90,0.95
9,Chaussure gauche,0.32,0.49
10,Chaussure droite,0.01,0.02
11,Visage,0.80,0.89
12,Jambe gauche,0.01,0.03
13,Jambe droite,0.02,0.04
15,Bras droit,0.82,0.90


**Lecture.** Chaussure droite 0,01, jambe gauche 0,01 : ce n'est pas que le modèle rate les chaussures. Il
**inverse** souvent la gauche et la droite (dans le corrigé, 59 % des pixels de chaussure droite sont prédits
« gauche »). Pour une marque, l'erreur est sans importance : le corrigé fusionne donc gauche et droite avant de
calculer l'IoU.

Les classes absentes de l'image (ni dans la vérité, ni dans la prédiction) n'ont pas d'IoU (`NaN`) : on ne peut ni
réussir ni rater ce qui n'existe pas. En revanche, une classe **prédite sans être présente** (une hallucination) a un
IoU de 0 : elle doit compter comme une erreur.

## ⚠️ Piège : « 96 % de pixels bien classés »

L'exactitude par pixel (pixels bien classés ÷ pixels totaux) semble naturelle. Comparons le modèle à un « modèle »
paresseux qui répond **fond** partout, sur les 50 images :

In [3]:
verites = [io.read_mask(m) for _, m in pairs]
predictions = [cache.read_prediction(i, "jpeg512") for i, _ in pairs]
cm_modele = sum(confusion_matrix(v, p) for v, p in zip(verites, predictions))
cm_paresseux = sum(confusion_matrix(v, np.zeros_like(v)) for v in verites)

for nom, c in [("modèle", cm_modele), ("tout en fond", cm_paresseux)]:
    iou = iou_per_class(c)
    print(f"{nom:<13} exactitude {pixel_accuracy(c):.2f} | IoU haut {iou[4]:.2f} | IoU pantalon {iou[6]:.2f}")

modèle        exactitude 0.96 | IoU haut 0.88 | IoU pantalon 0.90
tout en fond  exactitude 0.78 | IoU haut 0.00 | IoU pantalon 0.00


**Lecture.** Le « modèle » paresseux obtient déjà une exactitude de 0,78 sans voir un seul vêtement : le fond occupe
78 % des pixels. Son IoU sur le haut et le pantalon vaut 0, ce qui dit la vérité. Quand une classe domine, on mesure
**chaque classe séparément** (IoU par classe), puis on fait la moyenne des classes qui intéressent le métier.

## 3. Additionner les images avant de diviser

Pour 50 images, deux calculs sont possibles : moyenne des IoU de chaque image, ou IoU de la matrice **cumulée** sur
les 50 images. Le corrigé utilise le second : une ceinture de 30 pixels dans une image ne pèse pas autant qu'un
manteau de 30 000 pixels dans une autre.

## 4. Combien de hasard dans un score ?

Avec 50 autres photos, on aurait obtenu un autre score. Pour estimer cette variation sans nouvelles photos, on
**rejoue le tirage** : on pioche autant d'images qu'on en a, *avec remise* (certaines deux fois, d'autres pas du tout), on
recalcule le score, et on recommence. C'est le **bootstrap**. L'écart entre les tirages montre la marge d'incertitude.

In [4]:
from segvet import evaluation as ev

# Démonstration sur les 25 premières images seulement (le corrigé fait le calcul sur les 50)
cms = ev.image_confusions(verites[:25], predictions[:25])
rng = np.random.default_rng(0)
scores = [ev.clothing_miou(cms[rng.integers(0, 25, 25)].sum(axis=0)) for _ in range(200)]
print(f"Score sur ces 25 images : {ev.clothing_miou(cms.sum(axis=0)):.2f}")
print(f"200 tirages : de {min(scores):.2f} à {max(scores):.2f} ; 95 % entre {np.percentile(scores, 2.5):.2f} et {np.percentile(scores, 97.5):.2f}")

Score sur ces 25 images : 0.77
200 tirages : de 0.72 à 0.82 ; 95 % entre 0.73 et 0.81


On publie donc un score **avec sa fourchette**, plutôt qu'un chiffre seul. À toi de refaire le calcul sur les 50
images de ton notebook. Attention : cette marge ne couvre que le hasard du
choix des 50 images. Elle ne dit rien d'images **différentes** (d'autres influenceurs, d'autres cadrages), ni du fait
que le modèle a déjà vu ces 50 images (fiche 7).

### ✍️ À toi de jouer

Sur la grille du § 1, décale la prédiction `P` d'une colonne vers la gauche (`np.roll(P, -1, axis=1)`) et recalcule l'IoU. Puis fais la même expérience avec une silhouette fine, d'une seule colonne de large : `V2 = np.zeros((4, 4), bool); V2[:3, 1] = True`, décalée d'une colonne. Que remarques-tu ?

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
def iou(a, b):
    return (a & b).sum() / (a | b).sum()

print(iou(V, np.roll(P, -1, axis=1)))       # la silhouette large résiste au décalage

V2 = np.zeros((4, 4), bool); V2[:3, 1] = True
print(iou(V2, np.roll(V2, 1, axis=1)))      # 0.0 : décalée d'un pixel, la silhouette fine n'a plus rien en commun
```

**Pourquoi ?** un décalage d'un pixel ne change presque rien pour un objet large, mais peut tout ruiner pour un objet fin. C'est sans doute une des raisons pour lesquelles la ceinture (quelques pixels de large) a l'IoU le plus bas du projet.

</details>

## ✅ À retenir

1. IoU = pixels communs ÷ pixels couverts par l'une ou l'autre silhouette ; Dice = F1 sur les pixels.
2. L'exactitude par pixel est trompeuse quand le fond domine : un modèle « tout en fond » atteint 0,78 ici.
3. Classe absente partout : pas de score (`NaN`) ; classe inventée : IoU = 0.
4. Sur plusieurs images, on cumule la matrice de confusion avant de calculer l'IoU.
5. Les objets fins (ceinture, lunettes) sont pénalisés par le moindre décalage de contour.